# Week 17: LLM & Embeddings -- RAG, Prompt Engineering Basics
# 第 17 週：LLM 與嵌入應用 — 檢索增強、提示工程基礎

## 學習目標 Learning Objectives
1. 理解文字表示的演進：BoW → TF-IDF → Embeddings
2. 實作共現矩陣與 SVD 降維的簡易詞向量
3. 實作餘弦相似度 (Cosine Similarity) 並視覺化相似度矩陣
4. 使用 t-SNE/PCA 視覺化嵌入空間
5. 建立簡易 TF-IDF 搜尋引擎 (Semantic Search)
6. 理解 RAG 架構並實作簡易版本
7. 探索提示工程 (Prompt Engineering) 基礎技巧
8. 建立完整的文字分類 Pipeline
9. 在嵌入空間中進行文件聚類

**所需套件 Required packages:** numpy, pandas, matplotlib, scikit-learn, scipy

---

In [ ]:
# === Imports and Setup ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from scipy.spatial.distance import cosine as cosine_dist

# Plot settings
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

print('All packages imported successfully!')
print('\u6240\u6709\u5957\u4ef6\u532f\u5165\u6210\u529f\uff01')

---
## Part 1: Text Representation Evolution (BoW & TF-IDF)
## 第一部分：文字表示的演進 (BoW 與 TF-IDF)

文字表示經歷了從簡單的詞袋 (Bag-of-Words) 到 TF-IDF，再到密集向量嵌入 (Dense Embeddings) 的演進。

**Bag-of-Words (BoW):** Count word occurrences.

**TF-IDF (Term Frequency - Inverse Document Frequency):**

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$
$$
\text{IDF}(t) = \log\frac{N}{\text{df}(t)} + 1
$$

where $N$ is the total number of documents and $\text{df}(t)$ is the number of documents containing term $t$.

In [ ]:
# === Sample Document Corpus ===
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with many layers",
    "Natural language processing handles text and speech",
    "Computer vision processes images and videos",
    "Reinforcement learning trains agents through rewards",
    "Neural networks learn features from data automatically",
    "Text classification assigns labels to documents",
    "Image recognition identifies objects in pictures",
    "Speech recognition converts spoken words to text",
    "Recommendation systems suggest items to users",
]

categories = ['ML', 'DL', 'NLP', 'CV', 'RL', 'DL', 'NLP', 'CV', 'NLP', 'ML']

# BoW (Bag-of-Words)
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(documents)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print('=== Bag-of-Words (BoW) ===')
print(f'Vocabulary size: {len(bow_vectorizer.get_feature_names_out())}')
print(f'Document-Term Matrix shape: {bow_matrix.shape}')
print(f'\nTop 10 terms: {bow_vectorizer.get_feature_names_out()[:10]}')

print('\n=== TF-IDF ===')
print(f'Document-Term Matrix shape: {tfidf_matrix.shape}')
print(f'Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.2%}')

# Visualize Document-Term Matrix (first 5 docs, top 15 terms by TF-IDF)
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_dense = tfidf_matrix.toarray()

# Select top terms by mean TF-IDF
top_term_idx = np.argsort(tfidf_dense.mean(axis=0))[-15:]
top_terms = feature_names[top_term_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# BoW heatmap
bow_dense = bow_matrix.toarray()
im1 = axes[0].imshow(bow_dense[:, top_term_idx], cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(len(top_terms)))
axes[0].set_xticklabels(top_terms, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(len(documents)))
axes[0].set_yticklabels([f'Doc {i}' for i in range(len(documents))], fontsize=8)
axes[0].set_title('Bag-of-Words Matrix / \u8a5e\u888b\u77e9\u9663')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# TF-IDF heatmap
im2 = axes[1].imshow(tfidf_dense[:, top_term_idx], cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(top_terms)))
axes[1].set_xticklabels(top_terms, rotation=45, ha='right', fontsize=8)
axes[1].set_yticks(range(len(documents)))
axes[1].set_yticklabels([f'Doc {i}' for i in range(len(documents))], fontsize=8)
axes[1].set_title('TF-IDF Matrix / TF-IDF \u77e9\u9663')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.suptitle('BoW vs TF-IDF Document-Term Matrix / \u6587\u4ef6-\u8a5e\u5f59\u77e9\u9663\u6bd4\u8f03', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nKey Difference: TF-IDF downweights common terms (like "and", "to") and upweights distinctive terms.')

---
## Part 2: Word Embedding Concepts (Co-occurrence + SVD)
## 第二部分：詞嵌入概念 (共現矩陣 + SVD)

詞嵌入 (Word Embeddings) 將詞彙映射到低維密集向量空間。最簡單的方法是從共現矩陣 (Co-occurrence Matrix) 出發，再用 SVD 降維。

Word embeddings map words to dense vectors. A simple approach: build a co-occurrence matrix, then apply SVD.

$$
M_{\text{co-occ}} \approx U \Sigma V^T
$$

The rows of $U$ (truncated) serve as word vectors.

In [ ]:
# === Co-occurrence Matrix & SVD Word Vectors ===
# Build co-occurrence matrix with a window of 2
all_words = []
for doc in documents:
    all_words.extend(doc.lower().split())

vocab = sorted(set(all_words))
word2idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

# Build co-occurrence matrix
window_size = 2
cooccurrence = np.zeros((V, V))

for doc in documents:
    words = doc.lower().split()
    for i, w in enumerate(words):
        for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
            if i != j:
                cooccurrence[word2idx[w], word2idx[words[j]]] += 1

print(f'Vocabulary size: {V}')
print(f'Co-occurrence matrix shape: {cooccurrence.shape}')
print(f'Non-zero entries: {np.count_nonzero(cooccurrence)}')

# Apply SVD to get word vectors
n_components = 2  # For 2D visualization
svd = TruncatedSVD(n_components=n_components, random_state=42)
word_vectors = svd.fit_transform(cooccurrence)

print(f'\nSVD explained variance ratio: {svd.explained_variance_ratio_}')
print(f'Total variance explained: {svd.explained_variance_ratio_.sum():.2%}')

# Plot 2D word vectors
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Co-occurrence matrix heatmap (subset)
# Select interesting words
interesting_words = ['learning', 'neural', 'networks', 'text', 'image', 'data',
                     'language', 'vision', 'deep', 'machine', 'recognition', 'features']
interesting_idx = [word2idx[w] for w in interesting_words if w in word2idx]
interesting_labels = [vocab[i] for i in interesting_idx]

sub_matrix = cooccurrence[np.ix_(interesting_idx, interesting_idx)]
im = axes[0].imshow(sub_matrix, cmap='Blues')
axes[0].set_xticks(range(len(interesting_labels)))
axes[0].set_xticklabels(interesting_labels, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(len(interesting_labels)))
axes[0].set_yticklabels(interesting_labels, fontsize=8)
axes[0].set_title('Co-occurrence Matrix (subset)\n\u5171\u73fe\u77e9\u9663\uff08\u5b50\u96c6\uff09')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# 2D word vectors
ax2 = axes[1]
# Color words by category
word_categories = {}
ml_words = {'machine', 'learning', 'artificial', 'intelligence', 'data', 'features', 'agents', 'rewards'}
dl_words = {'deep', 'neural', 'networks', 'layers', 'learn'}
nlp_words = {'language', 'text', 'speech', 'words', 'natural', 'processing', 'classification', 'documents', 'labels'}
cv_words = {'computer', 'vision', 'images', 'videos', 'image', 'recognition', 'objects', 'pictures'}

for w in vocab:
    if w in dl_words:
        word_categories[w] = 'DL'
    elif w in nlp_words:
        word_categories[w] = 'NLP'
    elif w in cv_words:
        word_categories[w] = 'CV'
    elif w in ml_words:
        word_categories[w] = 'ML'
    else:
        word_categories[w] = 'Other'

color_map = {'ML': '#FF6B6B', 'DL': '#4ECDC4', 'NLP': '#45B7D1', 'CV': '#FFA07A', 'Other': '#CCCCCC'}
for i, w in enumerate(vocab):
    cat = word_categories.get(w, 'Other')
    c = color_map[cat]
    ax2.scatter(word_vectors[i, 0], word_vectors[i, 1], color=c, s=60, alpha=0.7, edgecolors='white')
    if w in interesting_words:
        ax2.annotate(w, (word_vectors[i, 0], word_vectors[i, 1]),
                    fontsize=8, ha='center', va='bottom',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

# Legend
for cat, col in color_map.items():
    if cat != 'Other':
        ax2.scatter([], [], color=col, label=cat, s=60)
ax2.legend(fontsize=9)
ax2.set_xlabel('SVD Component 1')
ax2.set_ylabel('SVD Component 2')
ax2.set_title('2D Word Vectors via SVD\nSVD \u8a5e\u5411\u91cf\u8996\u89ba\u5316')
ax2.grid(True, alpha=0.3)

plt.suptitle('Word Embeddings: Co-occurrence + SVD / \u8a5e\u5d4c\u5165\uff1a\u5171\u73fe\u77e9\u9663 + SVD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Cosine Similarity
## 第三部分：餘弦相似度

餘弦相似度衡量兩個向量的方向相似程度，是 NLP 中最常用的相似度指標。

Cosine similarity measures the directional similarity between two vectors.

$$
\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_{i=1}^{n} a_i b_i}{\sqrt{\sum_{i=1}^{n} a_i^2} \cdot \sqrt{\sum_{i=1}^{n} b_i^2}}
$$

Range: $[-1, 1]$ where 1 means identical direction, 0 means orthogonal, -1 means opposite.

In [ ]:
# === Cosine Similarity: From Scratch ===
def cosine_similarity_manual(a, b):
    """Compute cosine similarity between two vectors."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

# Demonstrate with simple example
v1 = np.array([1, 2, 3])
v2 = np.array([2, 4, 6])  # Same direction as v1
v3 = np.array([-1, -2, -3])  # Opposite direction
v4 = np.array([3, -1, 0])  # Different direction

print('=== Cosine Similarity Examples ===')
print(f'v1 = {v1}, v2 = {v2} (same direction)')
print(f'  cos(v1, v2) = {cosine_similarity_manual(v1, v2):.4f}')
print(f'v1 = {v1}, v3 = {v3} (opposite direction)')
print(f'  cos(v1, v3) = {cosine_similarity_manual(v1, v3):.4f}')
print(f'v1 = {v1}, v4 = {v4} (different direction)')
print(f'  cos(v1, v4) = {cosine_similarity_manual(v1, v4):.4f}')

# Compute document similarity matrix using TF-IDF
sim_matrix = cosine_similarity(tfidf_matrix)

# Visualize similarity matrix as heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=0, vmax=1)

# Add text annotations
for i in range(len(documents)):
    for j in range(len(documents)):
        val = sim_matrix[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

doc_labels = [f'D{i}:{cat}' for i, cat in enumerate(categories)]
ax.set_xticks(range(len(documents)))
ax.set_xticklabels(doc_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(documents)))
ax.set_yticklabels(doc_labels, fontsize=9)

plt.colorbar(im, ax=ax, shrink=0.8, label='Cosine Similarity')
ax.set_title('Document Similarity Matrix (TF-IDF + Cosine)\n\u6587\u4ef6\u76f8\u4f3c\u5ea6\u77e9\u9663', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Find most similar document pairs
print('\n=== Most Similar Document Pairs ===')
pairs = []
for i in range(len(documents)):
    for j in range(i+1, len(documents)):
        pairs.append((i, j, sim_matrix[i, j]))
pairs.sort(key=lambda x: x[2], reverse=True)
for i, j, sim in pairs[:5]:
    print(f'  Doc {i} <-> Doc {j}: similarity = {sim:.4f}')
    print(f'    "{documents[i][:50]}..."')
    print(f'    "{documents[j][:50]}..."')

---
## Part 4: Embedding Space Visualization (t-SNE / PCA)
## 第四部分：嵌入空間視覺化

使用 PCA 或 t-SNE 將高維 TF-IDF 向量投射到 2D，觀察文件的聚類結構。

We project high-dimensional TF-IDF vectors to 2D using PCA and t-SNE to observe document clusters.

- **PCA**: Linear projection preserving global structure
- **t-SNE**: Non-linear projection preserving local structure

In [ ]:
# === Embedding Space Visualization with PCA and t-SNE ===
# Expand corpus for better visualization
expanded_docs = documents + [
    "Supervised learning uses labeled training data",
    "Unsupervised learning discovers hidden patterns",
    "Convolutional neural networks excel at image tasks",
    "Recurrent neural networks process sequential data",
    "Sentiment analysis determines opinion in text",
    "Object detection locates items in images",
    "Clustering groups similar data points together",
    "Gradient descent optimizes model parameters",
    "Transfer learning reuses pretrained model knowledge",
    "Tokenization splits text into meaningful units",
]

expanded_categories = categories + ['ML', 'ML', 'CV', 'DL', 'NLP', 'CV', 'ML', 'ML', 'DL', 'NLP']

# Re-fit TF-IDF
tfidf_expanded = TfidfVectorizer()
X_tfidf = tfidf_expanded.fit_transform(expanded_docs)

# PCA projection
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tfidf.toarray())

# t-SNE projection
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
X_tsne = tsne.fit_transform(X_tfidf.toarray())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cat_colors = {'ML': '#FF6B6B', 'DL': '#4ECDC4', 'NLP': '#45B7D1', 'CV': '#FFA07A', 'RL': '#9370DB'}

for ax, X_proj, title in [(axes[0], X_pca, f'PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})'),
                           (axes[1], X_tsne, 't-SNE')]:
    for i, (x, y) in enumerate(X_proj):
        cat = expanded_categories[i]
        ax.scatter(x, y, color=cat_colors[cat], s=100, alpha=0.7, edgecolors='white', linewidth=1.5)
        ax.annotate(f'D{i}', (x, y), fontsize=7, ha='center', va='bottom')
    
    # Add legend
    for cat, col in cat_colors.items():
        ax.scatter([], [], color=col, label=cat, s=80)
    ax.legend(fontsize=9, loc='best')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.grid(True, alpha=0.3)

plt.suptitle('Embedding Space Visualization / \u5d4c\u5165\u7a7a\u9593\u8996\u89ba\u5316', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observation: Documents of the same category tend to cluster together in embedding space.')
print('\u89c0\u5bdf\uff1a\u76f8\u540c\u985e\u5225\u7684\u6587\u4ef6\u5728\u5d4c\u5165\u7a7a\u9593\u4e2d\u50be\u5411\u805a\u96c6\u5728\u4e00\u8d77\u3002')

---
## Part 5: Semantic Search with TF-IDF
## 第五部分：基於 TF-IDF 的語義搜尋

語義搜尋 (Semantic Search) 的核心是根據查詢與文件的相似度來排序結果。

Semantic search ranks documents by their similarity to the query.

$$
\text{score}(q, d) = \text{cos}(\text{TF-IDF}(q), \text{TF-IDF}(d))
$$

In [ ]:
# === Simple TF-IDF Search Engine ===
class SimpleSearchEngine:
    """A TF-IDF based document search engine.
    \u57fa\u65bc TF-IDF \u7684\u7c21\u6613\u641c\u5c0b\u5f15\u64ce\u3002
    """
    def __init__(self, documents):
        self.documents = documents
        self.vectorizer = TfidfVectorizer()
        self.doc_vectors = self.vectorizer.fit_transform(documents)
        print(f'Search engine initialized with {len(documents)} documents')
    
    def search(self, query, top_k=5):
        """Search for documents most similar to the query."""
        query_vector = self.vectorizer.transform([query])
        similarities = cosine_similarity(query_vector, self.doc_vectors).flatten()
        
        # Rank by similarity
        ranked_idx = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for rank, idx in enumerate(ranked_idx):
            if similarities[idx] > 0:
                results.append({
                    'rank': rank + 1,
                    'doc_id': idx,
                    'score': similarities[idx],
                    'text': self.documents[idx]
                })
        return results

# Initialize search engine
search_engine = SimpleSearchEngine(expanded_docs)

# Run several queries
queries = [
    "How do neural networks work?",
    "text analysis and classification",
    "image recognition and object detection",
    "training machine learning models",
]

for query in queries:
    print(f'\n{"="*60}')
    print(f'Query: "{query}"')
    print(f'{"="*60}')
    results = search_engine.search(query, top_k=3)
    for r in results:
        print(f'  #{r["rank"]} [score: {r["score"]:.4f}] Doc {r["doc_id"]}: {r["text"]}')

---
## Part 6: RAG (Retrieval-Augmented Generation) Concept
## 第六部分：RAG 檢索增強生成概念

RAG 結合檢索 (Retrieval) 與生成 (Generation)：先檢索相關文件，再將它們作為上下文提供給生成模型。

RAG combines retrieval and generation: first retrieve relevant documents, then provide them as context for the generator.

```
User Query --> [Retriever] --> Top-K Documents --> [Prompt Template] --> [LLM] --> Answer
```

Since we cannot call an LLM API in this notebook, we simulate the generation step with a template-based approach.

In [ ]:
# === RAG Pipeline: Retrieve + Augment + Generate (Template-based) ===

# Knowledge base
knowledge_base = [
    "Random Forest is an ensemble method that combines multiple decision trees using bagging.",
    "Gradient Boosting builds trees sequentially, each correcting errors of the previous one.",
    "Logistic Regression uses the sigmoid function to model binary classification probabilities.",
    "Support Vector Machines find the hyperplane that maximizes the margin between classes.",
    "K-Nearest Neighbors classifies based on the majority vote of the K closest training examples.",
    "Neural networks consist of layers of connected neurons with learnable weights and biases.",
    "Convolutional Neural Networks use convolutional filters to extract spatial features from images.",
    "Recurrent Neural Networks process sequences by maintaining hidden state across time steps.",
    "LSTM networks use gates (forget, input, output) to control information flow and handle long-range dependencies.",
    "The Transformer architecture uses self-attention to process all positions in parallel, eliminating sequential bottlenecks.",
    "Cross-validation divides data into K folds, training on K-1 and testing on 1, rotating through all folds.",
    "Overfitting occurs when a model performs well on training data but poorly on unseen test data.",
    "Feature engineering creates new input features to improve model performance and interpretability.",
    "Batch normalization normalizes layer inputs to stabilize and accelerate neural network training.",
    "Data augmentation artificially increases training data by applying transformations like rotation and flipping.",
]

# Initialize retriever
rag_search = SimpleSearchEngine(knowledge_base)

def rag_pipeline(question, search_engine, top_k=3):
    """Simple RAG pipeline: Retrieve -> Augment -> Generate (template-based)."""
    # Step 1: Retrieve
    retrieved = search_engine.search(question, top_k=top_k)
    
    # Step 2: Augment (build prompt)
    context = "\n".join([f"- {r['text']}" for r in retrieved])
    
    prompt = f"""Based on the following reference materials, answer the question.

Reference Materials:
{context}

Question: {question}

Answer:"""
    
    # Step 3: Generate (template-based, simulating LLM output)
    # In a real system, this would call an LLM API
    answer = f"Based on the retrieved information: {retrieved[0]['text']}"
    if len(retrieved) > 1:
        answer += f" Additionally, {retrieved[1]['text'].lower()}"
    
    return {
        'question': question,
        'retrieved_docs': retrieved,
        'prompt': prompt,
        'answer': answer
    }

# Demo RAG pipeline
questions = [
    "What is Random Forest?",
    "How does a neural network learn?",
    "What is the Transformer architecture?",
]

for q in questions:
    result = rag_pipeline(q, rag_search)
    print(f'\n{"="*70}')
    print(f'Question: {result["question"]}')
    print(f'\nRetrieved Documents:')
    for doc in result['retrieved_docs']:
        print(f'  [{doc["score"]:.3f}] {doc["text"]}')
    print(f'\nGenerated Answer:')
    print(f'  {result["answer"]}')

# Visualize RAG pipeline
print(f'\n{"="*70}')
print('Full prompt sent to LLM (for the last question):')
print(result['prompt'])

In [ ]:
# === RAG Architecture Diagram ===
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# Define boxes
boxes = [
    (1.5, 6, 2.5, 1.2, 'User Query\n\u4f7f\u7528\u8005\u63d0\u554f', '#FF6B6B'),
    (5.5, 6, 2.5, 1.2, 'Retriever\n\u6aa2\u7d22\u5668 (TF-IDF)', '#4ECDC4'),
    (5.5, 3.5, 2.5, 1.2, 'Knowledge Base\n\u77e5\u8b58\u5eab', '#FFD700'),
    (9.5, 6, 2.5, 1.2, 'Prompt Builder\n\u63d0\u793a\u7d44\u5408\u5668', '#87CEEB'),
    (9.5, 3.5, 2.5, 1.2, 'LLM Generator\n\u751f\u6210\u5668', '#9370DB'),
    (12.5, 3.5, 1.2, 1.2, 'Answer\n\u56de\u7b54', '#90EE90'),
]

for x, y, w, h, label, color in boxes:
    box = FancyBboxPatch((x - w/2, y - h/2), w, h,
                          boxstyle='round,pad=0.15', facecolor=color,
                          edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows
arrows = [
    (2.8, 6, 4.2, 6, 'query'),
    (5.5, 5.3, 5.5, 4.2, 'search'),
    (6.8, 6, 8.2, 6, 'top-K docs'),
    (9.5, 5.3, 9.5, 4.2, 'augmented\nprompt'),
    (10.8, 3.5, 11.9, 3.5, ''),
    (5.5, 4.8, 5.5, 5.3, ''),
]

for x1, y1, x2, y2, label in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
               arrowprops=dict(arrowstyle='->', color='#333', lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        offset = (0.2, 0) if x1 != x2 else (0.3, 0)
        ax.text(mx + offset[0], my + offset[1], label, fontsize=7, color='#666', style='italic')

ax.text(7, 7.5, 'RAG (Retrieval-Augmented Generation) Architecture',
        ha='center', fontsize=15, fontweight='bold')
ax.text(7, 1.5, 'In a real system, the LLM generates answers based on retrieved context,\n'
        'reducing hallucination and providing up-to-date information.',
        ha='center', fontsize=9, color='#666', style='italic')

plt.tight_layout()
plt.show()

---
## Part 7: Prompt Engineering Basics
## 第七部分：提示工程基礎

提示工程 (Prompt Engineering) 是設計有效提示以引導 LLM 產生期望輸出的技術。

Prompt engineering designs effective prompts to guide LLMs toward desired outputs.

**Three main strategies:**
1. **Zero-shot**: No examples, direct instruction
2. **Few-shot**: Provide examples for the model to learn from
3. **Chain-of-Thought (CoT)**: Guide step-by-step reasoning

In [ ]:
# === Prompt Engineering: Templates and Comparison ===

# Define prompt templates
prompt_templates = {
    'Zero-shot': {
        'description': 'Direct instruction without examples',
        'template': '''Classify the following text into one of these categories: 
Positive, Negative, Neutral.

Text: "{text}"
Category:''',
        'example_text': 'The movie was absolutely wonderful and I loved every minute of it!',
    },
    'Few-shot': {
        'description': 'Provide examples for in-context learning',
        'template': '''Classify the following text into: Positive, Negative, or Neutral.

Text: "I really enjoyed the food at this restaurant."
Category: Positive

Text: "The service was terrible and the wait was too long."
Category: Negative

Text: "The building is located on Main Street."
Category: Neutral

Text: "{text}"
Category:''',
        'example_text': 'The movie was absolutely wonderful and I loved every minute of it!',
    },
    'Chain-of-Thought': {
        'description': 'Guide step-by-step reasoning',
        'template': '''Classify the following text into: Positive, Negative, or Neutral.
Let's think step by step.

Text: "{text}"

Step 1: Identify key sentiment words in the text.
Step 2: Determine if the overall tone is positive, negative, or neutral.
Step 3: Provide the final classification.

Analysis:''',
        'example_text': 'The movie was absolutely wonderful and I loved every minute of it!',
    }
}

print('=== Prompt Engineering Strategies ===')
for name, info in prompt_templates.items():
    print(f'\n{"="*60}')
    print(f'Strategy: {name}')
    print(f'Description: {info["description"]}')
    print(f'{"="*60}')
    filled = info['template'].format(text=info['example_text'])
    print(filled)

# Visual comparison
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 15)
ax.set_ylim(0, 8)
ax.axis('off')

strategies = [
    ('Zero-shot', 'Direct instruction\nNo examples', '#FF6B6B',
     'Simple tasks\nClear instructions', 'Classify this text:\n"..."\nCategory:'),
    ('Few-shot', '2-5 examples\nprovided', '#4ECDC4',
     'Pattern learning\nComplex formats', 'Example 1: ... -> A\nExample 2: ... -> B\nNew: ... -> ?'),
    ('Chain-of-Thought', 'Step-by-step\nreasoning', '#FFD700',
     'Math/Logic\nComplex reasoning', 'Let\'s think step by step.\nStep 1: ...\nStep 2: ...'),
]

for i, (name, desc, color, use_case, example) in enumerate(strategies):
    x = 2.5 + i * 4.5
    # Title box
    box = FancyBboxPatch((x - 1.8, 5.5), 3.6, 1.5,
                          boxstyle='round,pad=0.15', facecolor=color,
                          edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(box)
    ax.text(x, 6.5, name, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(x, 5.9, desc, ha='center', va='center', fontsize=8)
    
    # Details box
    box2 = FancyBboxPatch((x - 1.8, 2.5), 3.6, 2.5,
                           boxstyle='round,pad=0.15', facecolor='#f0f0f0',
                           edgecolor=color, linewidth=2)
    ax.add_patch(box2)
    ax.text(x, 4.5, 'Best for:', ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(x, 3.8, use_case, ha='center', va='center', fontsize=8)
    ax.text(x, 3.0, example, ha='center', va='center', fontsize=7, family='monospace',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Arrow
    ax.annotate('', xy=(x, 5.4), xytext=(x, 5.1),
               arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

ax.text(7.5, 7.5, 'Prompt Engineering Strategies / \u63d0\u793a\u5de5\u7a0b\u7b56\u7565\u6bd4\u8f03',
        ha='center', fontsize=15, fontweight='bold')
ax.text(7.5, 1.5, 'Complexity increases from left to right, but so does effectiveness for complex tasks.',
        ha='center', fontsize=9, color='#666', style='italic')

plt.tight_layout()
plt.show()

---
## Part 8: Text Classification Pipeline
## 第八部分：文字分類 Pipeline

完整的文字分類 Pipeline：原始文字 → TF-IDF → 分類器 → 評估

Complete text classification pipeline: Raw Text -> TF-IDF -> Classifier -> Evaluation

In [ ]:
# === Text Classification Pipeline ===
# Create a larger synthetic dataset
np.random.seed(42)

# Templates for generating synthetic text data
tech_templates = [
    "The new {adj} algorithm improves {metric} by {pct}% on {dataset}",
    "{model} architecture achieves state-of-the-art results in {task}",
    "Researchers propose a novel {adj} method for {task} using {tech}",
    "This paper presents a {adj} approach to {task} with {model}",
    "We introduce {model} for efficient {task} on {dataset}",
]

ml_vocab = {'adj': ['supervised', 'ensemble', 'regularized', 'optimized', 'boosted'],
            'metric': ['accuracy', 'precision', 'recall', 'F1 score', 'AUC'],
            'pct': ['5', '10', '15', '3', '8'],
            'dataset': ['MNIST', 'CIFAR-10', 'ImageNet', 'GLUE', 'SQuAD'],
            'model': ['Random Forest', 'XGBoost', 'SVM', 'Linear Model', 'Decision Tree'],
            'task': ['classification', 'regression', 'clustering', 'prediction', 'detection'],
            'tech': ['cross-validation', 'feature engineering', 'hyperparameter tuning', 'bagging', 'boosting']}

dl_vocab = {'adj': ['deep', 'convolutional', 'recurrent', 'attention-based', 'transformer'],
            'metric': ['accuracy', 'BLEU', 'perplexity', 'mAP', 'loss'],
            'pct': ['12', '20', '25', '7', '15'],
            'dataset': ['ImageNet', 'COCO', 'WMT', 'LibriSpeech', 'OpenWebText'],
            'model': ['ResNet', 'BERT', 'GPT', 'ViT', 'LSTM'],
            'task': ['image recognition', 'text generation', 'translation', 'speech synthesis', 'segmentation'],
            'tech': ['self-attention', 'backpropagation', 'transfer learning', 'data augmentation', 'dropout']}

nlp_vocab = {'adj': ['semantic', 'contextual', 'multilingual', 'pretrained', 'fine-tuned'],
            'metric': ['BLEU', 'ROUGE', 'F1', 'accuracy', 'perplexity'],
            'pct': ['8', '14', '18', '6', '11'],
            'dataset': ['SQuAD', 'GLUE', 'WMT', 'CoNLL', 'SST-2'],
            'model': ['BERT', 'RoBERTa', 'T5', 'Transformer', 'Word2Vec'],
            'task': ['sentiment analysis', 'named entity recognition', 'question answering', 'summarization', 'translation'],
            'tech': ['tokenization', 'embeddings', 'attention mechanism', 'language modeling', 'fine-tuning']}

def generate_text(templates, vocab):
    template = np.random.choice(templates)
    filled = template
    for key in vocab:
        if '{' + key + '}' in filled:
            filled = filled.replace('{' + key + '}', np.random.choice(vocab[key]))
    return filled

# Generate dataset
n_per_class = 60
texts, labels = [], []
for label, vocab in [('ML', ml_vocab), ('DL', dl_vocab), ('NLP', nlp_vocab)]:
    for _ in range(n_per_class):
        texts.append(generate_text(tech_templates, vocab))
        labels.append(label)

texts = np.array(texts)
labels = np.array(labels)

# Shuffle
idx = np.random.permutation(len(texts))
texts, labels = texts[idx], labels[idx]

print(f'Generated dataset: {len(texts)} samples, {len(set(labels))} classes')
print(f'Class distribution: {dict(zip(*np.unique(labels, return_counts=True)))}')
print(f'\nSample texts:')
for i in range(3):
    print(f'  [{labels[i]}] {texts[i]}')

# Split
X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)

# Compare classifiers
classifiers = {
    'Naive Bayes': Pipeline([('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())]),
    'Logistic Regression': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Linear SVM': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LinearSVC(random_state=42, max_iter=2000))]),
    'Random Forest': Pipeline([('tfidf', TfidfVectorizer()), ('clf', RandomForestClassifier(n_estimators=100, random_state=42))]),
}

results = []
for name, pipe in classifiers.items():
    pipe.fit(X_train_txt, y_train_txt)
    train_acc = pipe.score(X_train_txt, y_train_txt)
    test_acc = pipe.score(X_test_txt, y_test_txt)
    cv = cross_val_score(pipe, texts, labels, cv=5, scoring='accuracy')
    results.append({'Classifier': name, 'Train Acc': train_acc, 'Test Acc': test_acc,
                   'CV Mean': cv.mean(), 'CV Std': cv.std()})

df_clf = pd.DataFrame(results).sort_values('Test Acc', ascending=False)
print('\n=== Text Classification Results ===')
print(df_clf.to_string(index=False, float_format='%.4f'))

# Detailed report for best classifier
best_clf_name = df_clf.iloc[0]['Classifier']
best_pipe = classifiers[best_clf_name]
y_pred_txt = best_pipe.predict(X_test_txt)

print(f'\n=== Best Classifier: {best_clf_name} ===')
print(classification_report(y_test_txt, y_pred_txt))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = axes[0].bar(df_clf['Classifier'], df_clf['Test Acc'], color=colors, edgecolor='white')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Text Classifier Comparison / \u6587\u5b57\u5206\u985e\u5668\u6bd4\u8f03')
axes[0].set_ylim(0.5, 1.05)
for bar, val in zip(bars, df_clf['Test Acc']):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}',
               ha='center', fontweight='bold', fontsize=10)
axes[0].tick_params(axis='x', rotation=15)

# Confusion matrix
cm = confusion_matrix(y_test_txt, y_pred_txt, labels=['DL', 'ML', 'NLP'])
im = axes[1].imshow(cm, cmap='Blues')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=14)
axes[1].set_xticks([0, 1, 2])
axes[1].set_xticklabels(['DL', 'ML', 'NLP'])
axes[1].set_yticks([0, 1, 2])
axes[1].set_yticklabels(['DL', 'ML', 'NLP'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].set_title(f'Confusion Matrix ({best_clf_name})\n\u6df7\u6dc6\u77e9\u9663')

plt.suptitle('Text Classification Pipeline Results / \u6587\u5b57\u5206\u985e\u7d50\u679c', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 9: Embedding Clustering
## 第九部分：嵌入空間聚類

在嵌入空間中對文件進行聚類 (Clustering)，可以發現文件的自然分組。

Clustering documents in embedding space reveals natural groupings.

In [ ]:
# === Embedding Clustering ===
# Use the full generated text dataset
tfidf_full = TfidfVectorizer(max_features=500)
X_full_tfidf = tfidf_full.fit_transform(texts)

# Apply KMeans clustering
n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_full_tfidf.toarray())

# Reduce to 2D for visualization
svd_2d = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd_2d.fit_transform(X_full_tfidf)

# Also get cluster centers in 2D
centers_2d = svd_2d.transform(kmeans.cluster_centers_)

# Evaluate clustering vs true labels
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(labels, cluster_labels)
nmi = normalized_mutual_info_score(labels, cluster_labels)

print(f'=== Clustering Evaluation ===')
print(f'Number of clusters: {n_clusters}')
print(f'Adjusted Rand Index (ARI): {ari:.4f}')
print(f'Normalized Mutual Info (NMI): {nmi:.4f}')

# Cross-tabulate clusters vs true labels
ct = pd.crosstab(labels, cluster_labels, rownames=['True'], colnames=['Cluster'])
print(f'\nCluster vs True Label Cross-tabulation:')
print(ct)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# True labels
true_color_map = {'ML': '#FF6B6B', 'DL': '#4ECDC4', 'NLP': '#45B7D1'}
for label_val in ['ML', 'DL', 'NLP']:
    mask = labels == label_val
    axes[0].scatter(X_2d[mask, 0], X_2d[mask, 1], color=true_color_map[label_val],
                   label=label_val, alpha=0.6, s=40, edgecolors='white')
axes[0].set_title('True Labels / \u771f\u5be6\u6a19\u7c64')
axes[0].legend()
axes[0].set_xlabel('SVD Component 1')
axes[0].set_ylabel('SVD Component 2')
axes[0].grid(True, alpha=0.3)

# Cluster labels
cluster_colors = ['#E74C3C', '#2ECC71', '#3498DB']
for c in range(n_clusters):
    mask = cluster_labels == c
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], color=cluster_colors[c],
                   label=f'Cluster {c}', alpha=0.6, s=40, edgecolors='white')
# Plot centers
axes[1].scatter(centers_2d[:, 0], centers_2d[:, 1], color='black', marker='X',
               s=200, linewidths=2, edgecolors='white', label='Centers', zorder=5)
axes[1].set_title(f'KMeans Clusters (ARI={ari:.3f})\nKMeans \u805a\u985e\u7d50\u679c')
axes[1].legend()
axes[1].set_xlabel('SVD Component 1')
axes[1].set_ylabel('SVD Component 2')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Embedding Clustering / \u5d4c\u5165\u7a7a\u9593\u805a\u985e', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nKey Insight: KMeans in TF-IDF space can discover document groups that align with true categories.')
print(f'ARI={ari:.3f} indicates {"strong" if ari > 0.5 else "moderate" if ari > 0.2 else "weak"} agreement with true labels.')

---
## Exercises / 練習題

### Exercise 1: Build a FAQ Retrieval System
### 練習 1：建立 FAQ 檢索系統

Build a FAQ system that:
- Stores question-answer pairs
- Given a user question, retrieves the most similar FAQ
- Returns the corresponding answer

In [ ]:
# === Exercise 1: FAQ Retrieval System ===

faq_data = [
    ("What is machine learning?", "Machine learning is a subset of AI that enables systems to learn from data."),
    ("How does deep learning differ from ML?", "Deep learning uses neural networks with many layers, while traditional ML uses simpler models."),
    ("What is overfitting?", "Overfitting occurs when a model memorizes training data and performs poorly on new data."),
    ("What is cross-validation?", "Cross-validation is a technique to evaluate models by partitioning data into training and testing sets multiple times."),
    ("What is a neural network?", "A neural network is a computing system inspired by biological neural networks, consisting of connected layers of nodes."),
    ("What is gradient descent?", "Gradient descent is an optimization algorithm that iteratively adjusts parameters to minimize a loss function."),
    ("What is a confusion matrix?", "A confusion matrix is a table showing true positives, false positives, true negatives, and false negatives."),
    ("What is transfer learning?", "Transfer learning applies knowledge from one task to improve performance on a different but related task."),
]

class FAQSystem:
    def __init__(self, faq_pairs):
        # TODO: Store FAQ pairs and build TF-IDF index on questions
        # Hint: Use TfidfVectorizer on the FAQ questions
        pass
    
    def ask(self, user_question, top_k=1):
        # TODO: Find the most similar FAQ question using cosine similarity
        # Return the corresponding answer(s)
        # Hint: Transform user_question with the same vectorizer, compute similarity
        pass

# Test:
# faq = FAQSystem(faq_data)
# answer = faq.ask("How can I prevent overfitting?")
# print(answer)
print('Exercise 1: Implement the FAQSystem class above.')

### Exercise 2: Compare BoW vs TF-IDF for Text Classification
### 練習 2：比較 BoW 與 TF-IDF 的分類效果

Using the generated text dataset from Part 8:
1. Train LogisticRegression with both BoW and TF-IDF features
2. Compare accuracy and F1 scores
3. Visualize the comparison

In [ ]:
# === Exercise 2: BoW vs TF-IDF Comparison ===

# TODO: Step 1 - Create BoW pipeline
# bow_pipe = Pipeline([('bow', CountVectorizer()), ('clf', LogisticRegression(max_iter=1000))])

# TODO: Step 2 - Create TF-IDF pipeline  
# tfidf_pipe = Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression(max_iter=1000))])

# TODO: Step 3 - Train both and compare on the test set
# Use X_train_txt, X_test_txt, y_train_txt, y_test_txt from Part 8

# TODO: Step 4 - Create a bar chart comparing the two approaches
# Compare accuracy, precision, recall, and F1 for each class

print('Exercise 2: Compare BoW vs TF-IDF classification performance.')
print('Expected: TF-IDF generally performs better because it downweights common terms.')

### Exercise 3: Simple Few-shot Prompt Template System
### 練習 3：簡易 Few-shot 提示模板系統

Build a `PromptBuilder` class that:
- Stores a system prompt
- Manages few-shot examples
- Generates formatted prompts for new inputs

In [ ]:
# === Exercise 3: Prompt Template System ===

class PromptBuilder:
    """A reusable few-shot prompt template builder."""
    def __init__(self, system_prompt, task_description):
        # TODO: Store system prompt and task description
        # Initialize empty examples list
        pass
    
    def add_example(self, input_text, output_text):
        # TODO: Add an input-output example pair
        pass
    
    def build_prompt(self, new_input):
        # TODO: Build the full prompt with:
        # 1. System prompt
        # 2. Task description
        # 3. All examples formatted as Input: ... Output: ...
        # 4. New input with Output: placeholder
        pass
    
    def __str__(self):
        # TODO: Return a summary of the prompt builder
        pass

# Test:
# builder = PromptBuilder(
#     system_prompt="You are a helpful assistant.",
#     task_description="Translate the following English text to French."
# )
# builder.add_example("Hello", "Bonjour")
# builder.add_example("Thank you", "Merci")
# prompt = builder.build_prompt("Good morning")
# print(prompt)
print('Exercise 3: Implement the PromptBuilder class above.')

---
## Summary / 教學重點整理

### Key Takeaways / 關鍵要點

| 主題 Topic | 重點 Key Point |
|------|------|
| Text Representation | BoW → TF-IDF → Dense Embeddings，從稀疏到密集的演進 |
| Word Embeddings | 共現矩陣 + SVD 可產生簡易詞向量，捕捉詞彙關係 |
| Cosine Similarity | 最常用的文本相似度指標，衡量向量方向\u800c\u975e\u5927\u5c0f |
| Embedding Visualization | PCA 保\u7559\u5168\u5c40\u7d50\u69cb\uff0ct-SNE \u4fdd\u7559\u5c40\u90e8\u7d50\u69cb |
| Semantic Search | \u4f7f\u7528\u5411\u91cf\u76f8\u4f3c\u5ea6\u6392\u5e8f\u6587\u4ef6\uff0c\u6bd4\u95dc\u9375\u5b57\u641c\u5c0b\u66f4\u667a\u6167 |
| RAG | \u6aa2\u7d22\u76f8\u95dc\u6587\u4ef6 + \u63d0\u4f9b\u7d66 LLM \u4f5c\u70ba\u4e0a\u4e0b\u6587\uff0c\u6e1b\u5c11\u5e7b\u89ba |
| Prompt Engineering | Zero-shot\u3001Few-shot\u3001CoT \u4e09\u5927\u7b56\u7565 |
| Text Classification | TF-IDF + \u5206\u985e\u5668\u662f\u7d93\u5178\u7684\u6587\u5b57\u5206\u985e Pipeline |
| Clustering | \u5728\u5d4c\u5165\u7a7a\u9593\u4e2d\u805a\u985e\u53ef\u767c\u73fe\u6587\u4ef6\u7684\u81ea\u7136\u5206\u7d44 |

### \u6838\u5fc3\u516c\u5f0f Core Formulas

**TF-IDF:**
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{df}(t)}$$

**Cosine Similarity:**
$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

---

### Next Week Preview / \u4e0b\u9031\u9810\u544a

**Week 18: Final Project Development & Presentation**
- \u5b8c\u6574\u7684 ML Pipeline \u6a21\u677f
- \u63a2\u7d22\u5f0f\u8cc7\u6599\u5206\u6790 (EDA) \u6a21\u677f
- \u6a21\u578b\u6bd4\u8f03\u6846\u67b6
- \u7d50\u679c\u8996\u89ba\u5316\u8207\u5831\u544a\u751f\u6210
- 18 \u9031\u8ab2\u7a0b\u56de\u9867